In [1]:
import sys
GRID_ENGINE_PARENT = "/kaggle/input/datasets/avranu01biswas/grid-engine"
if GRID_ENGINE_PARENT not in sys.path:
    sys.path.insert(0, GRID_ENGINE_PARENT)

import glob
import os
import time
import numpy as np
from concurrent.futures import ProcessPoolExecutor, as_completed
from grid_engine import build_grid_engine

INPUT_DIR = "/kaggle/input/datasets/dipangsu/dl-output-sih/SIH_Segmented_Seq08/"
OUTPUT_DIR = "./batch_grid_outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

all_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*_segmented.npy")))
target_files = all_files[1000:2000]  # The busy intersection!

print(f"Starting parallel processing and metric collection on {len(target_files)} frames...\n")

def process_single_frame(file_path):
    filename = os.path.basename(file_path)
    result = build_grid_engine(file_path, verbose=False)
    
    out_name = filename.replace("_segmented.npy", "_grid.npz")
    out_path = os.path.join(OUTPUT_DIR, out_name)
    
    np.savez_compressed(
        out_path,
        final_cells=result['final_cells'],
        grid_config=np.array(result['grid_config'])
    )
    
    # --- EXTRACT ALL METRICS ---
    n_input = result['stats']['n_input_points']
    assigned = result['final_cells']['point_count'].sum()
    n_cells = len(result['final_cells'])
    processing_time = result['timings']['total']
    
    # 1. Calculate hypothetical Uniform Grid size (for compression metric)
    gc = result['grid_config']
    grid_w = gc['grid_x_max'] - gc['grid_x_min']
    grid_h = gc['grid_y_max'] - gc['grid_y_min']
    finest_res = 0.0625 # 6.25 cm
    uniform_cells = int(np.ceil(grid_w / finest_res)) * int(np.ceil(grid_h / finest_res))
    
    # 2. Extract resolution distribution (how many 6cm cells vs 50cm cells)
    res_array = result['final_cells']['resolution']
    res_6cm = int(np.sum(np.isclose(res_array, 0.0625)))
    res_50cm = int(np.sum(np.isclose(res_array, 0.50)))
    
    return {
        'filename': filename,
        'n_input': n_input,
        'assigned': assigned,
        'n_cells': n_cells,
        'time': processing_time,
        'uniform_cells': uniform_cells,
        'res_6cm': res_6cm,
        'res_50cm': res_50cm
    }

# Track global metrics
start_time = time.time()
total_frames = len(target_files)
completed_frames = 0

global_points = 0
global_assigned = 0
global_cells = 0
global_engine_time = 0.0
global_uniform_cells = 0
global_res_6cm = 0
global_res_50cm = 0

with ProcessPoolExecutor() as executor:
    futures = [executor.submit(process_single_frame, f) for f in target_files]
    
    for future in as_completed(futures):
        stats = future.result()
        completed_frames += 1
        
        global_points += stats['n_input']
        global_assigned += stats['assigned']
        global_cells += stats['n_cells']
        global_engine_time += stats['time']
        global_uniform_cells += stats['uniform_cells']
        global_res_6cm += stats['res_6cm']
        global_res_50cm += stats['res_50cm']
        
        if completed_frames % 10 == 0 or completed_frames == 1 or completed_frames == total_frames:
            print(f"[{completed_frames} / {total_frames}] Processed {stats['filename']}")

# Calculate Final Averages
end_time = time.time()
avg_time = (global_engine_time / total_frames) * 1000
avg_retention = (global_assigned / global_points) * 100 if global_points > 0 else 0
avg_points_sec = (global_points / global_engine_time) if global_engine_time > 0 else 0
cell_reduction_pct = (1.0 - (global_cells / global_uniform_cells)) * 100

pct_6cm = (global_res_6cm / global_cells) * 100
pct_50cm = (global_res_50cm / global_cells) * 100

print(f"\n=======================================================")
print(f" GLOBAL BATCH METRICS (Over {total_frames} Frames)")
print(f"=======================================================")
print(f"1. DATA COMPRESSION")
print(f"   Cell Reduction vs Uniform Grid : {cell_reduction_pct:.2f}% reduction")
print(f"   Average Final Cells Per Frame  : {global_cells / total_frames:,.0f} cells\n")

print(f"2. REAL-TIME SPEED")
print(f"   Average Grid Engine Speed      : {avg_time:.1f} ms per frame")
print(f"   Grid Engine Throughput         : {avg_points_sec:,.0f} points / second\n")

print(f"3. INFORMATION RETENTION")
print(f"   Total Raw Points Processed     : {global_points:,}")
print(f"   Average Point Retention        : {avg_retention:.3f}%\n")

print(f"4. SMART RESOLUTION DISTRIBUTION")
print(f"   Ultra-High Res (6cm) Cells     : {pct_6cm:.1f}% of map")
print(f"   Low Res Terrain (50cm) Cells   : {pct_50cm:.1f}% of map")
print(f"=======================================================\n")
print(f"Files saved in: {OUTPUT_DIR}")

Starting parallel processing and metric collection on 1000 frames...

[1 / 1000] Processed 001003_segmented.npy
[10 / 1000] Processed 001009_segmented.npy
[20 / 1000] Processed 001019_segmented.npy
[30 / 1000] Processed 001029_segmented.npy
[40 / 1000] Processed 001039_segmented.npy
[50 / 1000] Processed 001048_segmented.npy
[60 / 1000] Processed 001057_segmented.npy
[70 / 1000] Processed 001070_segmented.npy
[80 / 1000] Processed 001079_segmented.npy
[90 / 1000] Processed 001088_segmented.npy
[100 / 1000] Processed 001099_segmented.npy
[110 / 1000] Processed 001110_segmented.npy
[120 / 1000] Processed 001119_segmented.npy
[130 / 1000] Processed 001128_segmented.npy
[140 / 1000] Processed 001139_segmented.npy
[150 / 1000] Processed 001149_segmented.npy
[160 / 1000] Processed 001159_segmented.npy
[170 / 1000] Processed 001169_segmented.npy
[180 / 1000] Processed 001179_segmented.npy
[190 / 1000] Processed 001189_segmented.npy
[200 / 1000] Processed 001199_segmented.npy
[210 / 1000] Proc